# Day 28 — Feature engineering
Objectives:
- Encoding (one-hot, ordinal), scaling, binning.
- Missing flags and interaction terms.
- sklearn.compose + ColumnTransformer basics.

<!-- BEGIN BEGINNER NOTEBOOK DEEP DIVE -->
## How to use this notebook

This is the editable learner artifact for `python-28`. Read
`python/ds-60day/companion-guides/day28_feature_engineering.md` first, then work here with the **Python (ds60sqlpy)**
kernel. Restart the kernel and run from top to bottom so an earlier
hidden value cannot make later code appear correct.

For every example: (1) write a prediction, (2) run the cell,
(3) compare the exact value, type, shape, rows, or side effect with
the stated observation, and (4) explain one mismatch before moving
on. For every exercise, use its dedicated work cell and include a
real assertion or bounded inspection. The official solution stays
closed until you have a tested attempt.

The notebook is deliberately offline after course setup. Do not add
`%pip`, credentials, absolute developer paths, or shell-specific
setup here. If an import fails, use the repository doctor and the
catalog from a terminal rather than changing only this kernel.

## Core mental model

A feature is a model input available at the moment a prediction would be
made. Feature engineering changes representation to expose useful
structure: scaling numbers, encoding categories, extracting dates,
binning, or combining fields. It must preserve the time/knowledge
boundary—post-outcome data and target-derived values are leakage.

A transformer learns parameters during `fit` and applies them during
`transform`. Split evaluation data before fitting learned means, scales,
bins, imputers, or category vocabularies. A pipeline binds every learned
step to the model and applies identical logic at inference. Define
missing, unseen-category, and zero-variance behavior.

### Vocabulary

- **feature:** an input value available to a model at prediction time.
- **transformer:** an object that fits parameters and transforms data.
- **fit:** learn transformation/model parameters from training data.
- **transform:** apply already learned parameters to data.
- **pipeline:** an ordered fitted chain of preprocessing and estimation.
- **leakage:** evaluation or future/target information entering training features.

## Syntax anatomy

A `ColumnTransformer` selects named column groups and runs separate
pipelines, then concatenates their outputs. `Pipeline([...])` fits each
preprocessing step only on the training rows passed to `fit` and passes
the transformed output to the estimator. `handle_unknown="ignore"`
defines inference behavior for categories absent during fit.

### Worked example 1 — Fit scaling on training data only

The held-out value is transformed with training parameters. Before running the next cell, predict its final displayed
value and identify the line responsible for every intermediate.

In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler

train = np.array([[1.0], [2.0], [3.0]])
test = np.array([[10.0]])
scaler = StandardScaler().fit(train)
(scaler.mean_.tolist(), scaler.transform(test).round(2).tolist())

**Expected observation:** The training mean is `[2.0]`; the held-out value becomes a large positive standardized value. Test data did not influence the mean.

If your result differs, compare inputs and types before rerunning.
Then explain the example from input to evidence in your own words.

### Worked example 2 — Handle an unseen category explicitly

The encoder's inference contract belongs inside preprocessing. Predict first; then run the next cell.

In [ ]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
encoder.fit([["red"], ["blue"]])
encoded = encoder.transform([["green"], ["red"]])
(encoder.categories_[0].tolist(), encoded.tolist())

**Expected observation:** Known levels are `['blue', 'red']`; unseen `green` becomes all zeros while `red` activates its learned column.

## Debugging clinic

When evidence differs from your prediction, use this order:

1. Write when every candidate feature becomes known relative to prediction time.
2. Split before fitting any stateful transformation.
3. Inspect transformed feature names, shape, and output ordering.
4. Test missing values, unseen categories, and zero variance through the fitted pipeline—not ad hoc notebook fixes.

**Alternative to compare:** Stateless arithmetic/date extraction can be a small function transformer; learned preprocessing belongs in fitted pipeline objects.

**Boundary to test:** Unseen categories, missing values, zero variance, target/time leakage, sparse/dense memory growth, and feature-name drift require tests.

Do not move on merely because the cell runs. Explain which object,
branch, axis, row, or resource changed and why.

In [ ]:
import pandas as pd, seaborn as sns
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

df = sns.load_dataset('titanic').dropna(subset=['sex','class','fare','survived'])
X = df[['sex','class','fare']]
y = df['survived']
pre = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), ['sex','class']),
    ('num', StandardScaler(), ['fare'])
])
pipe = Pipeline([('pre', pre), ('clf', LogisticRegression(max_iter=1000))])
pipe.fit(X,y)
pipe.score(X,y)


## Exercises and progressive hints

Each item is a complete mini-contract. Before writing code, copy its input,
expected behavior, constraints, and verification into your work cell. A
result is not complete merely because it “looks right”; run the stated
assertion or inspection and explain what it proves.

1. Add a fare-bin feature using a transformer inside the preprocessing pipeline. **Contract:** thresholds are learned or fixed without evaluation data, bin closure/labels are documented, and missing/out-of-range behavior is defined.
   **Verify:** inspect fitted thresholds/feature names and test exact boundaries plus one missing value.

2. Compare one model pipeline with scaling against the same model pipeline without scaling. **Constraints:** keep split/folds, random state, features, model, and all other transforms identical; fit both only on training data.
   **Expected behavior:** report held-out or cross-validated scores with uncertainty, not training score.
   **Verify:** report both score estimates, uncertainty intervals, and their difference; compare that difference with a stated practical threshold and connect the result to the model family's sensitivity to feature scale.

### Additional mastery practice

Fit learned transformations only on training data and keep every feature inside a reproducible pipeline with explicit unknown/missing behavior.

Continue with five new exercises. Record each prediction before running
code; these extend rather than replace the original practice above.

3. **Prediction:** Predict how fitting a scaler or category encoder before the train/test split leaks information from evaluation data.
   **Progressive hint:** Learned means, scales, and categories become evaluation-derived parameters.
   **Verify:** Fit both ways on a fixture with an extreme held-out value; record differing learned parameters and assert the pipeline fit uses training rows only.
4. **Tracing:** Trace numeric and categorical columns through a `ColumnTransformer` and state the output order/shape.
   **Progressive hint:** Each branch selects columns, transforms them, then outputs are concatenated.
   **Verify:** Inspect `get_feature_names_out()` and transformed shape; map every output column back to its numeric or categorical branch in the declared order.
5. **Implementation:** Add deterministic date features (weekday and month) without retaining the original target or post-outcome timestamp.
   **Progressive hint:** Validate timezone and the moment at which a feature becomes known.
   **Verify:** Assert weekday/month values for known timestamps and prove target/post-outcome columns are absent from the transformed feature names.
6. **Debugging:** Repair a pipeline that scales one-hot indicator columns unnecessarily and fails on an unseen category.
   **Progressive hint:** Use separate branches and `handle_unknown='ignore'`.
   **Verify:** Pass an unseen category through the repaired pipeline and assert no failure, expected output shape, and no scaling step applied to one-hot columns.
7. **Edge case and explanation:** Handle a zero-variance numeric feature, unseen category, and missing value at inference; define tests for each.
   **Progressive hint:** The fitted pipeline—not ad hoc notebook code—owns these policies.
   **Verify:** Run three inference fixtures—zero variance, unseen category, missing value—through the same fitted pipeline and assert each documented result.

Before opening the reference solution, write one sentence explaining
which contract or mental model each result confirms.

### Practice 1 — prediction, attempt, and evidence

**Contract reminder:** Add a fare-bin feature using a transformer inside the preprocessing pipeline. **Contract:** thresholds are learned or fixed without evaluation data, bin closure/labels are documented, and missing/out-of-range behavior is defined. **Verify:** inspect fitted thresholds/feature names and test exact boundaries plus one missing value.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 1 — your work
# Short contract: Add a fare-bin feature using a transformer inside the preprocessing pipeline. thresholds are learned or fixed without evaluation data, bin closure/labels are documented, and mis...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 2 — prediction, attempt, and evidence

**Contract reminder:** Compare one model pipeline with scaling against the same model pipeline without scaling. **Constraints:** keep split/folds, random state, features, model, and all other transforms identical; fit both only on training data. **Expected behavior:** report held-out or cross-validated scores with uncertainty, not training score. **Verify:** report both score estimates, uncertainty intervals, and their difference; compare that difference with a stated practical threshold and connect the result to the model family's sensitivity to feature scale.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 2 — your work
# Short contract: Compare one model pipeline with scaling against the same model pipeline without scaling. keep split/folds, random state, features, model, and all other transforms identical; fit...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 3 — prediction, attempt, and evidence

**Contract reminder:** **Prediction:** Predict how fitting a scaler or category encoder before the train/test split leaks information from evaluation data. **Progressive hint:** Learned means, scales, and categories become evaluation-derived parameters. **Verify:** Fit both ways on a fixture with an extreme held-out value; record differing learned parameters and assert the pipeline fit uses training rows only.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 3 — your work
# Short contract: Predict how fitting a scaler or category encoder before the train/test split leaks information from evaluation data. Learned means, scales, and categories become evaluation-deri...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 4 — prediction, attempt, and evidence

**Contract reminder:** **Tracing:** Trace numeric and categorical columns through a `ColumnTransformer` and state the output order/shape. **Progressive hint:** Each branch selects columns, transforms them, then outputs are concatenated. **Verify:** Inspect `get_feature_names_out()` and transformed shape; map every output column back to its numeric or categorical branch in the declared order.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 4 — your work
# Short contract: Trace numeric and categorical columns through a `ColumnTransformer` and state the output order/shape. Each branch selects columns, transforms them, then outputs are concatenated...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 5 — prediction, attempt, and evidence

**Contract reminder:** **Implementation:** Add deterministic date features (weekday and month) without retaining the original target or post-outcome timestamp. **Progressive hint:** Validate timezone and the moment at which a feature becomes known. **Verify:** Assert weekday/month values for known timestamps and prove target/post-outcome columns are absent from the transformed feature names.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 5 — your work
# Short contract: Add deterministic date features (weekday and month) without retaining the original target or post-outcome timestamp. Validate timezone and the moment at which a feature becomes...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 6 — prediction, attempt, and evidence

**Contract reminder:** **Debugging:** Repair a pipeline that scales one-hot indicator columns unnecessarily and fails on an unseen category. **Progressive hint:** Use separate branches and `handle_unknown='ignore'`. **Verify:** Pass an unseen category through the repaired pipeline and assert no failure, expected output shape, and no scaling step applied to one-hot columns.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 6 — your work
# Short contract: Repair a pipeline that scales one-hot indicator columns unnecessarily and fails on an unseen category. Use separate branches and `handle_unknown='ignore'`. Pass an unseen catego...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 7 — prediction, attempt, and evidence

**Contract reminder:** **Edge case and explanation:** Handle a zero-variance numeric feature, unseen category, and missing value at inference; define tests for each. **Progressive hint:** The fitted pipeline—not ad hoc notebook code—owns these policies. **Verify:** Run three inference fixtures—zero variance, unseen category, missing value—through the same fitted pipeline and assert each documented result.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 7 — your work
# Short contract: Handle a zero-variance numeric feature, unseen category, and missing value at inference; define tests for each. The fitted pipeline—not ad hoc notebook code—owns these policies....
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):
